# Week 4: tile-direct 100 m train (10×10 AE grid)

Train on the **1:1 balanced 100 m tiles** (`tiles_balanced_min5/min20`), not on pooled cell scores.

**Input** per tile: up to 10×10 AlphaEarth 10 m cells (64-D each), missing slots zero-filled, plus a 10×10 occupancy mask → **6500-D**.

**Heads** (shared LORO folds):
1. Ridge multi-output on `(frac_res, frac_nres)`
2. Logistic regression on `y ∈ {NRES, RES}`

Compare to the cell-pool baseline that scores the same tiles with `mean(p_res) ≥ 0.50` from the 10 m LR (`outputs/week4_100m_tiles_balanced/`).

```bash
.venv/bin/python -u scripts/train_week4_100m_tiles_loro.py --min-cells 5
.venv/bin/python -u scripts/train_week4_100m_tiles_loro.py --min-cells 20
```

### min ≥ 5 (primary)

| Metric | Value |
|---|---:|
| Tiles | 52,492 |
| OOF macro-F1 (LR) | 0.719 |
| LORO F1 mean ± std | 0.700 ± 0.063 |
| OOF mean MAE (Ridge) | 0.448 |
| Cell-pool F1 @ 0.50 | 0.791 |
| Δ F1 (tile-direct − pool) | -0.071 |
| Backend | gpu · 8 GPUs / 80 CPUs |
| Runtime | 271.4 s |


![LORO F1 by metro](../../outputs/week4_100m_tile_train/figures/loro_f1_by_region_min5.png)

![LORO MAE by metro](../../outputs/week4_100m_tile_train/figures/loro_mae_by_region_min5.png)

![OOF true vs pred](../../outputs/week4_100m_tile_train/figures/oof_true_vs_pred_min5.png)

![OOF confusion](../../outputs/week4_100m_tile_train/figures/oof_confusion_min5.png)


### min ≥ 20

| Metric | Value |
|---|---:|
| Tiles | 34,494 |
| OOF macro-F1 (LR) | 0.726 |
| LORO F1 mean ± std | 0.705 ± 0.053 |
| OOF mean MAE (Ridge) | 0.401 |
| Cell-pool F1 @ 0.50 | 0.808 |
| Δ F1 (tile-direct − pool) | -0.082 |

![LORO F1 by metro, min≥20](../../outputs/week4_100m_tile_train/figures/loro_f1_by_region_min20.png)

![LORO MAE by metro, min≥20](../../outputs/week4_100m_tile_train/figures/loro_mae_by_region_min20.png)

![OOF true vs pred, min≥20](../../outputs/week4_100m_tile_train/figures/oof_true_vs_pred_min20.png)

![OOF confusion, min≥20](../../outputs/week4_100m_tile_train/figures/oof_confusion_min20.png)


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path("../..").resolve()
OUT = ROOT / "outputs" / "week4_100m_tile_train"
summary = json.loads((OUT / "summary.json").read_text())
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "paths"} for k, v in summary.items()}, indent=2))
for tag in (5, 20):
    p = OUT / f"loro_metrics_min{tag}.csv"
    if p.exists():
        print(p.name)
        display(pd.read_csv(p))


## Paths

| Artifact | Path |
|---|---|
| OOF preds | `outputs/week4_100m_tile_train/oof_predictions_min{5,20}.parquet` |
| Per-fold metrics | `outputs/week4_100m_tile_train/loro_metrics_min{5,20}.csv` |
| Checkpoints | `outputs/week4_100m_tile_train/checkpoints/` |
| Figures | `outputs/week4_100m_tile_train/figures/` |
| Train script | `scripts/train_week4_100m_tiles_loro.py` |
